# SAR surface-water mapping

**Default: supplied first-100 profile.** The user-supplied recipe selects the first 100 ordered rows of the official upstream manual training CSV. This notebook acquires and audits those exact pairs, verifies source metadata and freezes the existing seed-42 event split: 29 train (Mekong/Nigeria/Pakistan), 40 validation (India), 31 test (Ghana). [Exact supplied selection](https://github.com/kaustubh-dot/sar-water-mapping/blob/implementation/remaining-phases/reports/supplied_subset_recipe.md).

The supplied profile retains honest prior project exposure: 22 identifiers were used by earlier development training, including 8 current test identifiers. Existing development checkpoints cannot establish unseen-input performance. The alternative `upstream-development` profile preserves the earlier official 60/20/20 experiment; its tables remain development results. The generic organizer override still stops before calibration.

The `full` profile is the complete **classical** experiment, including source checkout, installation, bounded data access, audit, scientific checks, train-only looks, validation, scoring, figures and export. Neural training is disabled. Remote GPU training has separate runtime and provenance; inference time does not establish full training reproduction within 30 minutes. Observed surface water does not distinguish new flooding from permanent water. Fresh uncached Colab execution remains unverified.


In [ ]:
from time import perf_counter
NOTEBOOK_START = perf_counter()  # Run first: includes setup and documented access below.
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, platform, subprocess, sys, shutil
STAGES = {}
STARTED_UTC = datetime.now(timezone.utc).isoformat()
if sys.version_info < (3, 12):
    raise RuntimeError('Python >=3.12 for the pinned Colab dependencies is required. Select a compatible Colab runtime.')


The default downloads only the exact supplied 100 pairs into `data/supplied-v1`, with pinned object generations, byte-integrity checks and label-blind event membership frozen before object access. It uses the separately inspected `reports/supplied_training_rois-v1.csv` for effective looks and `reports/supplied_heldout_enl_rois-v1.csv` only for held-out ENL. Source CSV, recipe, selection, acquisition, inventory and audit identities are verified before calibration. This run records its existing TIFF count separately from the immutable original acquisition log. Raw data and checkpoints stay outside the export bundle.

Set `DATA_SOURCE='upstream-development'` to reproduce the earlier committed selection and its original ROI records. The generic `organizer` override requires its own long-form inventory, training ROI CSV and audited `ORGANIZER_DATA_CONFIG`; it remains unsupported for calibration and stops explicitly. Coordinates use half-open `y0:y1,x0:x1`; ROI columns are `patch_id,y0,y1,x0,x1,suitability,roi_class`. Record suitability from raw imagery before inspecting filtered results.

Upload/mount work performed before the first executable cell must be disclosed separately. No credentials belong in the notebook. Use a clean checkout; set a reviewed commit SHA for exact reproduction.


In [ ]:
REPOSITORY = 'https://github.com/kaustubh-dot/sar-water-mapping.git'
REPO_REF = 'implementation/remaining-phases'  # Use a reviewed commit SHA for exact reproduction.
REPO = Path('/content/sar-water-mapping') if 'google.colab' in sys.modules else Path.cwd()
DATA_SOURCE = 'supplied-first100'  # Alternatives: 'upstream-development', unsupported generic 'organizer'.
INVENTORY = None  # Organizer override: Path('/content/.../inventory.csv').
SUPPLIED_SPLITS = None  # Optional organizer patch_id,split CSV; upstream membership is verified below.
TRAINING_ROIS = None  # Supplied default: reports/supplied_training_rois-v1.csv; development retains its recorded ROIs.
HELDOUT_ENL_ROIS = None  # Supplied default: reports/supplied_heldout_enl_rois-v1.csv.
ORGANIZER_DATA_CONFIG = None  # Organizer-only audited source/provenance configuration.
DATA_ACCESS_DESCRIPTION = 'bounded supplied first100 manual pairs; actual cache/access conditions recorded below'
EXTERNAL_UPLOAD_OR_MOUNT_SECONDS = None  # Measured separately if access began before this notebook.
PROFILE = 'full'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


In [ ]:
def checkout_source(repository, reference, destination):
    """Resolve a branch, tag or full commit SHA and verify this clean checkout."""
    destination = Path(destination)
    if not reference or reference != reference.strip():
        raise ValueError('Supply an explicit branch, tag or full commit SHA.')
    existing_checkout = destination.exists()
    if not existing_checkout:
        subprocess.run(['git', 'clone', '--no-checkout', repository, str(destination)], check=True)
    elif not (destination / '.git').exists():
        raise RuntimeError('Existing source directory is not a Git checkout.')
    prefix = ['git', '-C', str(destination)]
    actual_remote = subprocess.check_output(prefix + ['remote', 'get-url', 'origin'], text=True).strip()
    if actual_remote != repository:
        raise RuntimeError('Existing checkout origin does not match the requested repository.')
    if existing_checkout and subprocess.check_output(prefix + ['status', '--porcelain'], text=True).strip():
        raise RuntimeError('Existing checkout has uncommitted changes; preserve them and use a clean destination.')
    # Fetching the requested reference also supports a reachable full commit SHA.
    subprocess.run(prefix + ['fetch', '--no-tags', 'origin', reference], check=True)
    expected = subprocess.check_output(prefix + ['rev-parse', '--verify', 'FETCH_HEAD^{commit}'], text=True).strip()
    if len(reference) == 40 and all(c in '0123456789abcdefABCDEF' for c in reference):
        if expected.lower() != reference.lower():
            raise RuntimeError('Fetched commit does not match the requested full SHA.')
    subprocess.run(prefix + ['checkout', '--detach', expected], check=True)
    actual = subprocess.check_output(prefix + ['rev-parse', 'HEAD'], text=True).strip()
    if actual != expected:
        raise RuntimeError('Executed checkout does not match the resolved requested reference.')
    return actual

checkout_start = perf_counter()
SOURCE_SHA = checkout_source(REPOSITORY, REPO_REF, REPO)
os.chdir(REPO)
print('Requested source reference:', REPO_REF)
print('Verified executed source commit:', SOURCE_SHA)
STAGES['source_checkout_seconds'] = perf_counter() - checkout_start
setup_start = perf_counter()
lock = REPO / 'requirements-colab.txt'
if not lock.is_file():
    raise FileNotFoundError('The tested core dependency lock requirements-colab.txt is missing from this checkout.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(lock)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
# Editable-install .pth hooks affect newly started Python processes, not this kernel.
# Make the verified checkout importable immediately without restarting the timer.
SOURCE_ROOT = (REPO / 'src').resolve()
if not (SOURCE_ROOT / 'sar_water/__init__.py').is_file():
    raise FileNotFoundError('The reviewed sar_water source package is missing from this checkout.')
sys.path.insert(0, str(SOURCE_ROOT))
# Keep Colab's preinstalled torch/CUDA untouched; this classical profile does not use them.
STAGES['installation_seconds'] = perf_counter() - setup_start


In [ ]:
from sar_water.data import sha256_file
from urllib.parse import urlsplit

def freeze_upstream_selection(selection_source, data_root):
    """Reuse only the exact committed, label-blind 100-pair selection."""
    from collections import Counter
    selection_source, data_root = Path(selection_source), Path(data_root)
    frozen = json.loads(selection_source.read_text(encoding='utf-8'))
    selected = frozen['selected']
    counts = dict(Counter(row['split'] for row in selected))
    if len(selected) != 100 or counts != {'train': 60, 'validation': 20, 'test': 20}:
        raise ValueError('The upstream profile requires the locked 100-pair 60/20/20 selection.')
    if len({row['patch_id'] for row in selected}) != 100:
        raise ValueError('Duplicate identifier in frozen upstream selection.')
    for row in selected:
        if (row['image_name'] != row['patch_id'] + '_S1Hand.tif'
                or row['mask_name'] != row['patch_id'] + '_LabelHand.tif'):
            raise ValueError('Frozen upstream selection must contain only matching hand-labeled pairs.')
    data_root.mkdir(parents=True, exist_ok=True)
    target = data_root / 'selection.json'
    if target.exists() and sha256_file(target) != sha256_file(selection_source):
        raise ValueError('Cached upstream selection changed; use a fresh data directory matching the committed selection.')
    if not target.exists():
        target.write_bytes(selection_source.read_bytes())
    return target

def verify_upstream_inputs(selection_path, data_root, inventory_path, split_path):
    """Verify pinned official lists and every inventory row before split freezing."""
    from collections import Counter
    from io import StringIO
    from urllib.parse import urlsplit
    from urllib.request import urlopen
    selection_path, data_root, inventory_path, split_path = map(Path, (selection_path, data_root, inventory_path, split_path))
    provenance = json.loads(selection_path.read_text(encoding='utf-8'))
    selected = provenance['selected']
    counts = dict(Counter(row['split'] for row in selected))
    if len(selected) != 100 or counts != {'train': 60, 'validation': 20, 'test': 20}:
        raise ValueError('Only the locked 100-pair upstream development profile is accepted.')
    official, official_hashes = {}, {}
    for split in ('train', 'validation', 'test'):
        source = provenance['official_splits'][split]
        local = data_root / Path(urlsplit(source['url']).path).name
        if not local.exists():
            with urlopen(source['url'], timeout=90) as response:
                local.write_bytes(response.read())
        if sha256_file(local) != source['sha256']:
            raise ValueError('Pinned official source-list hash changed: ' + split)
        pairs = [tuple(row) for row in csv.reader(StringIO(local.read_text(encoding='utf-8'))) if row]
        if len(pairs) != source['upstream_pair_count']:
            raise ValueError('Pinned official source-list pair count changed.')
        for image, mask in pairs:
            if image in official:
                raise ValueError('Duplicate image across official source lists.')
            official[image] = (mask, split)
        official_hashes[split] = source['sha256']
    membership, expected = {}, {}
    for row in selected:
        if row['patch_id'] in membership or official.get(row['image_name']) != (row['mask_name'], row['split']):
            raise ValueError('Frozen pair does not match official membership.')
        membership[row['patch_id']] = row['split']
        for kind, name, folder in (('image', row['image_name'], 'S1Hand'), ('mask', row['mask_name'], 'LabelHand')):
            expected[(row['patch_id'], kind)] = (name, folder, row['split'])
    with inventory_path.open(newline='', encoding='utf-8-sig') as stream:
        inventory = list(csv.DictReader(stream))
    seen = set()
    for row in inventory:
        key = (row['patch_id'], row['kind'])
        if key in seen or key not in expected:
            raise ValueError('Unexpected or duplicate upstream inventory row.')
        seen.add(key)
        name, folder, split = expected[key]
        path = Path(row['path'])
        path = path if path.is_absolute() else inventory_path.parent / path
        if (path.resolve() != (data_root/name).resolve() or row['split'] != split
                or row['units'] != 'db' or row['band_order'] != 'VV,VH' or row['label_source'] != 'manual'
                or row['source'] != 'gs://sen1floods11/v1.1/data/flood_events/HandLabeled/' + folder + '/' + name):
            raise ValueError('Upstream inventory must retain exact official manual pairs, paths, units and membership.')
    if seen != set(expected):
        raise ValueError('Upstream inventory must contain exactly 200 rows for the 100 frozen pairs.')
    with split_path.open('x', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(['patch_id', 'split'])
        writer.writerows(sorted(membership.items()))
    return dict(source='upstream-development', organizer_subset=False, pair_count=100,
        split_counts=counts, selection_sha256=sha256_file(selection_path),
        inventory_sha256=sha256_file(inventory_path), official_membership_verified=True,
        official_source_list_sha256=official_hashes,
        provenance_limitation='Correlated official-subset development benchmark: events span partitions and seven chip pairs touch across partitions. No held-out-event, unseen-geography or independent-sample statistical generalization claim.')

access_start = perf_counter()
if DATA_SOURCE not in ('supplied-first100', 'upstream-development', 'organizer'):
    raise ValueError('Select supplied-first100, upstream-development or the explicit organizer override.')
RUN_ROOT = REPO / 'outputs' / ('colab-' + RUN_ID)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
INPUTS = RUN_ROOT / 'inputs'
INPUTS.mkdir()
AUDIT, CALIBRATION = RUN_ROOT / 'audit', RUN_ROOT / 'calibration'
FROZEN_CONFIG, RESULTS = RUN_ROOT / 'experiment.json', RUN_ROOT / 'results'
SOURCE_CONFIG = None
if DATA_SOURCE == 'supplied-first100':
    if any(value is not None for value in (INVENTORY, SUPPLIED_SPLITS, ORGANIZER_DATA_CONFIG)):
        raise ValueError('The supplied profile uses only its exact recipe inventory and internal frozen splits.')
    data_root = REPO / 'data/supplied-v1'
    # The immutable completion log can describe an earlier acquisition. Record this run too.
    existing_tiff_count_before_helper = len(list(data_root.glob('*.tif'))) if data_root.exists() else 0
    download_start = perf_counter()
    acquisition = RUN_ROOT / 'reports/supplied_acquisition'
    subprocess.run([sys.executable, str(REPO / 'scripts/prepare_supplied_data.py'), '--data-root', str(data_root), '--output', str(acquisition)], check=True)
    STAGES['supplied_download_and_object_verification_seconds'] = perf_counter() - download_start
    INVENTORY = data_root / 'inventory.csv'
    SUPPLIED_SPLITS = data_root / 'splits.csv'
    TRAINING_ROIS = TRAINING_ROIS or REPO / 'reports/supplied_training_rois-v1.csv'
    HELDOUT_ENL_ROIS = HELDOUT_ENL_ROIS or REPO / 'reports/supplied_heldout_enl_rois-v1.csv'
    metadata_snapshot = INPUTS / 'supplied_acquisition'
    metadata_snapshot.mkdir()
    for name in ('flood_train_data.csv', 'supplied_subset_selection.csv', 'selection.json', 'download.json', 'split_freeze.json', 'splits.csv', 'inventory.csv'):
        shutil.copy2(data_root / name, metadata_snapshot / name)
    for name in ('supplied_subset_recipe.json', 'supplied_subset_selection.csv'):
        shutil.copy2(REPO / 'reports' / name, INPUTS / name)
    download = json.loads((data_root / 'download.json').read_text(encoding='utf-8'))
    DATA_PROVENANCE = dict(source='supplied-first100', organizer_subset=False,
        existing_tiff_count_before_helper=existing_tiff_count_before_helper,
        acquisition_completion_log_origin_counts=download['origin_counts'],
        acquisition_completion_log_note='Immutable log describes its original acquisition, not this notebook execution.',
        split_counts=download['split_counts'], previous_training_exposure_by_split=download['previous_training_exposure_by_split'],
        provenance_limitation='New internal event split; 22 prior project training exposures include 8 current test IDs. Source bindings are checked after raster audit.')
    DATA_ACCESS_DESCRIPTION = 'exact first100 upstream manual training rows; seed42 internal event split; existing TIFF count before helper=' + str(existing_tiff_count_before_helper)
elif DATA_SOURCE == 'upstream-development':
    if any(value is not None for value in (INVENTORY, SUPPLIED_SPLITS, TRAINING_ROIS, HELDOUT_ENL_ROIS, ORGANIZER_DATA_CONFIG)):
        raise ValueError('For organizer inputs select DATA_SOURCE=organizer; upstream uses only its frozen inventory and recorded ROIs.')
    data_root = REPO / 'data/development-v1'
    selection = freeze_upstream_selection(REPO / 'reports/development_data_selection.json', data_root)
    download_start = perf_counter()
    # Keep this execution's download log in the bundle, preserving source reports.
    subprocess.run([sys.executable, str(REPO / 'scripts/prepare_development_data.py'), '--data-root', str(data_root)], check=True, cwd=RUN_ROOT)
    STAGES['upstream_download_and_object_verification_seconds'] = perf_counter() - download_start
    INVENTORY = data_root / 'inventory.csv'
    SUPPLIED_SPLITS = INPUTS / 'supplied_splits.csv'
    DATA_PROVENANCE = verify_upstream_inputs(selection, data_root, INVENTORY, SUPPLIED_SPLITS)
    TRAINING_ROIS = REPO / 'reports/classical_training_rois.csv'
    HELDOUT_ENL_ROIS = REPO / 'reports/heldout_enl_rois.csv'
    for source in [selection] + [data_root / Path(urlsplit(value['url']).path).name
                              for value in json.loads(selection.read_text())['official_splits'].values()]:
        shutil.copy2(source, INPUTS / source.name)
    DATA_ACCESS_DESCRIPTION = 'bounded 100 upstream manual pairs at pinned GCS generations; official 60/20/20 membership; not organizer subset'
else:
    if INVENTORY is None or TRAINING_ROIS is None or ORGANIZER_DATA_CONFIG is None:
        raise ValueError('Organizer override requires its own inventory, training ROIs and audited ORGANIZER_DATA_CONFIG.')
    organizer_config = json.loads(Path(ORGANIZER_DATA_CONFIG).read_text(encoding='utf-8'))
    if organizer_config['real_data'].get('organizer_subset') is not True:
        raise ValueError('Organizer source configuration must explicitly identify organizer_subset=true.')
    DATA_PROVENANCE = dict(source='organizer', organizer_subset=True,
        source_config_sha256=sha256_file(ORGANIZER_DATA_CONFIG),
        provenance_limitation=organizer_config['real_data'].get('provenance_limitation', 'not yet supplied; audit required'))
    shutil.copy2(ORGANIZER_DATA_CONFIG, INPUTS / 'organizer_source_config.json')
for name, value in (('inventory', INVENTORY), ('training_rois', TRAINING_ROIS)):
    if not Path(value).is_file():
        raise FileNotFoundError(name + ': ' + str(value))
INVENTORY = Path(INVENTORY).resolve()
training_snapshot = INPUTS / 'training_rois.csv'
shutil.copy2(TRAINING_ROIS, training_snapshot)
TRAINING_ROIS = training_snapshot
if HELDOUT_ENL_ROIS is not None:
    heldout_snapshot = INPUTS / 'heldout_enl_rois.csv'
    shutil.copy2(HELDOUT_ENL_ROIS, heldout_snapshot)
    HELDOUT_ENL_ROIS = heldout_snapshot
DATA_PROVENANCE.update(inventory_sha256=sha256_file(INVENTORY), training_rois_sha256=sha256_file(TRAINING_ROIS),
    heldout_enl_rois_sha256=sha256_file(HELDOUT_ENL_ROIS) if HELDOUT_ENL_ROIS else None)
(INPUTS / 'data_provenance.json').write_text(json.dumps(DATA_PROVENANCE, indent=2) + '\n', encoding='utf-8')
STAGES['documented_data_access_seconds'] = perf_counter() - access_start  # Includes the download substage.
print(json.dumps(DATA_PROVENANCE, indent=2))


In [ ]:
checks_start = perf_counter()
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_data.py', 'tests/test_metrics.py', 'tests/test_filters.py', 'tests/test_segment.py', 'tests/test_benchmark.py', 'tests/test_evaluate.py', 'tests/test_notebook.py', 'tests/test_development_data.py', 'tests/test_calibration.py', 'tests/test_supplied_data.py', 'tests/test_supplied_profile.py', '--basetemp', str(RUN_ROOT / 'pytest')], check=True)
STAGES['scientific_checks_seconds'] = perf_counter() - checks_start


In [ ]:
audit_start = perf_counter()
command = [sys.executable, '-m', 'sar_water.data', str(INVENTORY), '--output', str(AUDIT), '--contact-sheet']
if SUPPLIED_SPLITS is not None:
    command.extend(['--splits', str(Path(SUPPLIED_SPLITS).resolve())])
subprocess.run(command, check=True)
MANIFEST, SPLITS = AUDIT / 'manifest.csv', AUDIT / 'splits.csv'
# The audit writer automatically serializes POSIX paths relative to the manifest.
with MANIFEST.open(newline='', encoding='utf-8-sig') as stream:
    manifest_rows = list(csv.DictReader(stream))
if any(Path(row[name]).is_absolute() or '\\' in row[name] for row in manifest_rows for name in ('image_path', 'mask_path')):
    raise ValueError('Expected a portable relative-path manifest.')
with SPLITS.open(newline='', encoding='utf-8-sig') as stream:
    membership = {row['patch_id']: row['split'] for row in csv.DictReader(stream)}
for roi_path, expected_split in ((TRAINING_ROIS, 'train'), (HELDOUT_ENL_ROIS, 'test')):
    if roi_path is not None:
        with roi_path.open(newline='', encoding='utf-8-sig') as stream:
            rois = list(csv.DictReader(stream))
        if not rois or any(membership.get(row['patch_id']) != expected_split for row in rois):
            raise ValueError('ROI membership must be ' + expected_split + ' only.')
if DATA_SOURCE == 'supplied-first100':
    from sar_water.supplied_profile import write_supplied_source_config
    SOURCE_CONFIG = INPUTS / 'supplied_source_config.json'
    write_supplied_source_config(MANIFEST, SPLITS, data_root, SOURCE_CONFIG,
        recipe_path=INPUTS / 'supplied_subset_recipe.json', selection_path=INPUTS / 'supplied_subset_selection.csv')
    DATA_PROVENANCE['source_config_sha256'] = sha256_file(SOURCE_CONFIG)
    (INPUTS / 'data_provenance.json').write_text(json.dumps(DATA_PROVENANCE, indent=2) + '\n', encoding='utf-8')
STAGES['audit_seconds'] = perf_counter() - audit_start
from IPython.display import display, Image
display(Image(filename=str(AUDIT / 'contact_sheet.png')))


Calibration uses training ROIs for effective looks and validation masks for the common channel/threshold rule, bounded filter parameters and selected filtered input. It creates a new versioned configuration bound to manifest, split, training-ROI and looks-artifact identities. The full per-band validation table and separate cleanup ablation are retained. Final evaluation refuses unfrozen or mismatched inputs. Otsu may split unimodal dry land into false water; constant images use the recorded calibrated fallback.

The supplied profile additionally freezes a metadata-only source configuration after the raster audit. Calibration verifies its full source/selection/acquisition/split chain before source access and again before freezing choices, then carries its path, SHA-256, membership and prior training exposure into the experiment configuration. The notebook verifies those bindings before opening held-out inputs. Snapshots of ROIs, acquisition metadata, recipe and source configuration remain in the run bundle. No configuration or calibration result is rewritten after freezing.


In [ ]:
def verify_calibration_identity(config_path, manifest_path, splits_path, training_roi_path, validation_path, *, source_config_path=None):
    """Check the disk identities consumed by paired training and final evaluation."""
    config_path = Path(config_path)
    real = json.loads(config_path.read_text(encoding='utf-8'))['real_data']
    if real.get('split_frozen') is not True or real.get('calibration_frozen') is not True:
        raise ValueError('Split and calibration must be frozen before test access.')
    inputs = dict(manifest=Path(manifest_path), splits=Path(splits_path), training_roi=Path(training_roi_path))
    hashes = {}
    for name, path in inputs.items():
        if not real.get(name + '_sha256') or sha256_file(path) != real[name + '_sha256']:
            raise ValueError(name + ' hash does not match frozen calibration.')
        bound = Path(real[name + '_path'])
        bound = bound if bound.is_absolute() else config_path.parent / bound
        if bound.resolve() != path.resolve():
            raise ValueError(name + ' path does not match frozen calibration.')
        hashes[name] = real[name + '_sha256']
    looks = Path(real['looks_artifact_path'])
    looks = looks if looks.is_absolute() else config_path.parent / looks
    if sha256_file(looks) != real['looks_artifact_sha256']:
        raise ValueError('Training looks artifact hash does not match frozen calibration.')
    frozen_looks = json.loads(looks.read_text(encoding='utf-8'))
    if any(frozen_looks.get(name + '_sha256') != value for name, value in hashes.items()):
        raise ValueError('Training looks artifact is bound to different input hashes.')
    source_hash = None
    if real.get('source') == 'supplied-first100' or source_config_path is not None:
        from sar_water.supplied_profile import verify_supplied_source_config
        if source_config_path is None:
            raise ValueError('Supplied source configuration required before held-out access.')
        source_config_path = Path(source_config_path)
        source_hash = sha256_file(source_config_path)
        if real.get('source_config_sha256') != source_hash:
            raise ValueError('Supplied source configuration hash mismatch.')
        if (config_path.parent / real['source_config_path']).resolve() != source_config_path.resolve():
            raise ValueError('Supplied source configuration path mismatch.')
        source_real = verify_supplied_source_config(source_config_path, manifest_path, splits_path)
        for name, value in source_real.items():
            if name in ('manifest_path', 'splits_path'):
                continue  # The frozen experiment binds these relative to its own directory above.
            if name == 'source_artifacts':
                for key, binding in value.items():
                    frozen_binding = real[name][key]
                    if (frozen_binding['sha256'] != binding['sha256'] or
                            (config_path.parent / frozen_binding['path']).resolve() != (source_config_path.parent / binding['path']).resolve()):
                        raise ValueError('Supplied source artifact binding mismatch.')
            elif real.get(name) != value:
                raise ValueError('Supplied source provenance mismatch: ' + name)
    validation = json.loads(Path(validation_path).read_text(encoding='utf-8'))
    if source_hash is not None and validation.get('source_config_sha256') != source_hash:
        raise ValueError('Validation source configuration hash mismatch.')
    if validation.get('config_sha256') != sha256_file(config_path):
        raise ValueError('Validation config hash does not match the executed frozen config.')
    if validation.get('input_sha256') != hashes or validation.get('test_files_opened') != 0:
        raise ValueError('Validation evidence must bind the same inputs without test access.')
    return dict(config_file_sha256=sha256_file(config_path), manifest_file_sha256=hashes['manifest'],
        splits_file_sha256=hashes['splits'], training_roi_sha256=hashes['training_roi'],
        looks_artifact_sha256=real['looks_artifact_sha256'], **({'source_config_sha256':source_hash} if source_hash is not None else {}))

calibration_start = perf_counter()
calibrator = REPO / 'scripts/calibrate_classical.py'
if not calibrator.is_file():
    raise FileNotFoundError('Reviewed validation calibration implementation is missing from this checkout.')
if DATA_SOURCE == 'organizer':
    raise RuntimeError('Generic organizer inventory was audited, but an arbitrary supplied-set source profile is unsupported. Use DATA_SOURCE=supplied-first100 only for the exact frozen recipe. No organizer test scoring has run.')
command = [sys.executable, str(calibrator), '--manifest', str(MANIFEST), '--splits', str(SPLITS), '--rois', str(TRAINING_ROIS), '--config', str(FROZEN_CONFIG), '--output', str(CALIBRATION)]
if SOURCE_CONFIG is not None:
    command.extend(['--source-config', str(SOURCE_CONFIG)])
subprocess.run(command, check=True)
CALIBRATION_IDENTITY = verify_calibration_identity(FROZEN_CONFIG, MANIFEST, SPLITS, TRAINING_ROIS, CALIBRATION / 'validation.json', source_config_path=SOURCE_CONFIG)
STAGES['training_roi_and_validation_calibration_seconds'] = perf_counter() - calibration_start
frozen = json.loads(FROZEN_CONFIG.read_text())
print(json.dumps(frozen['real_data'], indent=2))
print('Frozen identities for evaluation or separate paired training:', json.dumps(CALIBRATION_IDENTITY, indent=2))


In [ ]:
from sar_water.evaluate import run_evaluation
if PROFILE != 'full':
    raise ValueError('This notebook executes the full classical profile. Add separately verified neural checkpoint provenance before claiming checkpoint evaluation.')
evaluation_start = perf_counter()
runtime = run_evaluation(MANIFEST, SPLITS, FROZEN_CONFIG, RESULTS,
    enl_rois=HELDOUT_ENL_ROIS, profile=PROFILE,
    runtime_type='colab_cpu' if 'google.colab' in sys.modules else 'local_cpu_not_colab',
    external_setup_seconds=evaluation_start-NOTEBOOK_START, synthetic=True, figures=True)
STAGES['frozen_real_synthetic_evaluation_export_seconds'] = perf_counter() - evaluation_start


In [ ]:
display_start = perf_counter()
def show_csv(path):
    from IPython.display import HTML, display
    import html
    with Path(path).open(newline='', encoding='utf-8') as stream:
        rows = list(csv.reader(stream))
    display(HTML('<div style="overflow:auto"><table>' + ''.join('<tr>' + ''.join('<td style="padding:4px;border:1px solid #ddd">' + html.escape(value) + '</td>' for value in row) + '</tr>' for row in rows) + '</table></div>'))
print('Real held-out surface-water segmentation; primary cleanup disabled')
show_csv(RESULTS / 'results_real.csv')
print('Synthetic held-out PSNR; simulated known references, never real SAR ground truth')
show_csv(RESULTS / 'results_synthetic.csv')
print('ENL:', runtime['enl_status'])
show_csv(RESULTS / 'enl_summary.csv')
figure_paths = sorted((RESULTS / 'figures').glob('*.png'))
for figure_path in figure_paths[:8]:
    display(Image(filename=str(figure_path)))

STAGES['result_table_and_figure_display_seconds'] = perf_counter() - display_start


Inspect the aligned VV/VH images, errors and raw/filtered power ratio for lost narrow water and boundaries; report failures even if pooled scores improve. Per-patch predictions never alter the scoring mask. ENL reports undefined ROI counts and uses fixed positive-power regions; a smoother image is not necessarily a more faithful one.

The exports retain the unchanged raw baseline, primary four-row table, separate cleanup ablation, pooled counts, fixed water-positive macro set, dry-patch false positives, per-band radiometric changes, synthetic selection/test rows and provenance. A shorter Colab neural training schedule would be a separate experiment; no neural accuracy or remote-training time is implied here.
The authoritative timing sidecar includes the one final ZIP write and archive checksum. It records source checkout, installation, download/object verification, audit, scientific checks, calibration, real/synthetic evaluation/export and display durations. The data-access stage includes its download substage, so stage values are not additive. The notebook must still be executed in a fresh Colab runtime and its actual access conditions recorded before claiming the under-30-minute gate.


In [ ]:
from sar_water.data import sha256_file

def export_bundle(run_root, evidence, notebook_start, stages):
    """Archive once; save authoritative post-archive timing in a sibling sidecar."""
    export_start = perf_counter()
    run_root = Path(run_root)
    archive_base = str(run_root.parent / run_root.name)
    timing_path = run_root.parent / (run_root.name + '.runtime.json')
    manifest = dict(evidence, stages_seconds=dict(stages),
        authoritative_timing_sidecar=timing_path.name,
        timing_note='The sibling sidecar is written after the final archive and hash; it is intentionally outside this archive.')
    (run_root / 'bundle_manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    archive = shutil.make_archive(archive_base, 'zip', run_root)
    archive_hash = sha256_file(archive)
    stages['bundle_export_seconds'] = perf_counter() - export_start
    final = dict(evidence, stages_seconds=dict(stages), archive_path=str(archive),
        archive_sha256=archive_hash, authoritative_timing_sidecar=timing_path.name,
        total_wall_seconds=perf_counter() - notebook_start,
        timing_scope='First executable cell through the only archive write and archive checksum; excludes only timing-sidecar serialization and subsequent display.')
    final['measured_below_30_minutes'] = final['total_wall_seconds'] < 1800
    timing_path.write_text(json.dumps(final, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    return archive, timing_path, final

notebook_evidence = dict(started_utc=STARTED_UTC, profile=PROFILE, neural_training_enabled=False,
    runtime_type=runtime['runtime_type'], source_commit_sha=SOURCE_SHA, requested_source_ref=REPO_REF,
    dependency_lock_sha256=sha256_file(REPO / 'requirements-colab.txt'),
    inventory_sha256=sha256_file(INVENTORY), training_rois_sha256=sha256_file(TRAINING_ROIS),
    data_source=DATA_SOURCE, data_provenance=DATA_PROVENANCE, calibration_identity=CALIBRATION_IDENTITY,
    heldout_enl_rois_sha256=sha256_file(HELDOUT_ENL_ROIS) if HELDOUT_ENL_ROIS else None,
    data_access=DATA_ACCESS_DESCRIPTION, external_upload_or_mount_seconds=EXTERNAL_UPLOAD_OR_MOUNT_SECONDS,
    stages_note='documented_data_access_seconds includes upstream_download_and_object_verification_seconds; stages are not additive',
    fresh_runtime_attestation='unverified; record actual fresh runtime and access conditions before submission')
archive, timing_path, notebook_evidence = export_bundle(RUN_ROOT, notebook_evidence, NOTEBOOK_START, STAGES)
print(json.dumps(notebook_evidence, indent=2))
print('Bundle:', archive)
print('Authoritative final timing sidecar (outside the ZIP):', timing_path)
from IPython.display import FileLink
display(FileLink(archive))
display(FileLink(str(timing_path)))
